# Subset allele metadata

In [1]:
import polars as pl
import pandas as pd

In [2]:
meta_inputs = "../1_inputs"
meta_outputs = "../3_outputs"

## VarChAMP Allele QC

Check the consistency between nucleotide change and its respective AA change

In [3]:
allele_df = pl.scan_csv(f"{meta_outputs}/varchamp_consolidated_alleles_clinvar_gnomad_sequence_confirmed.tsv", separator="\t", infer_schema_length=10000)
allele_df = allele_df.collect()
allele_df

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,chr,chr_pos_38,mutation_id_rc4,mutation_id_cegs2,…,nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean,gnomad_af
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,f64,i64,str,i64,str,i64,str,i64,str,str,i64,str,str,…,str,str,str,str,str,str,str,str,str,i64,i64,str,str,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,f64
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""",2,"""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""2""",0,"""NULL""",0,"""NULL""",0,"""NULL""","""1""",155240033,"""6""","""NULL""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""",2,"""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""1""",0,"""NULL""",0,"""NULL""",0,"""NULL""","""1""",155238225,"""73""","""NULL""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""",2,"""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""7""",0,"""NULL""",0,"""NULL""",0,"""NULL""","""1""",155237453,"""113""","""NULL""",…,"""-""","""RCV000004573|RCV000079357|RCV0…","""MONDO:MONDO:0011945,MedGen:C18…","""Gaucher disease perinatal leth…","""germline;unknown""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155237453,155237453,"""1q22""","""criteria provided, multiple su…",15,"""-""","""N""","""ClinGen:CA221417,UniProtKB:P04…",3,4328,155237453,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000232587|SCV000321701|SCV0…","""-""","""-""","""887G>A ""","""Arg296Gln""","""NM_000157.4""",2,"""1_Pathogenic""",null
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""",2,"""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1.0,1,"""2""

In [4]:
## QC
from Bio.Seq import Seq
aa_single_to_three = {
    'A': 'Ala',  # Alanine
    'R': 'Arg',  # Arginine
    'N': 'Asn',  # Asparagine
    'D': 'Asp',  # Aspartic acid
    'C': 'Cys',  # Cysteine
    'Q': 'Gln',  # Glutamine
    'E': 'Glu',  # Glutamic acid
    'G': 'Gly',  # Glycine
    'H': 'His',  # Histidine
    'I': 'Ile',  # Isoleucine
    'L': 'Leu',  # Leucine
    'K': 'Lys',  # Lysine
    'M': 'Met',  # Methionine
    'F': 'Phe',  # Phenylalanine
    'P': 'Pro',  # Proline
    'S': 'Ser',  # Serine
    'T': 'Thr',  # Threonine
    'W': 'Trp',  # Tryptophan
    'Y': 'Tyr',  # Tyrosine
    'V': 'Val'   # Valine
}

for row in allele_df.iter_rows(named=True): #.filter(pl.col("hgvs_nt").is_duplicated()).sort(by="hgvs_nt")
    # hgvs_df = dat_info.filter(pl.col("hgvs_nt")==hgvs_nt)
    nuc_pos = int(row["nt_change"][:-3])
    nuc_r, nuc_v = row["nt_change"].split(">")[0][-1], row["nt_change"].split(">")[-1][-1]

    nuc_var = row["sequence"][nuc_pos-1] ## row["ref_cds"][nuc_pos-1], 
    nuc_seq = row["sequence"]
    # print(nuc_r, nuc_v, nuc_ref, nuc_var)
    prot_pos = int(row["aa_change"][3:-3])
    prot_r, prot_v = row["aa_change"][:3], row["aa_change"][-3:]
    prot_ref_seq = Seq(nuc_seq[:nuc_pos-1]+nuc_r+nuc_seq[nuc_pos:]).translate()
    prot_var_seq = Seq(nuc_seq).translate()
    try:
        prot_ref, prot_var = aa_single_to_three[prot_ref_seq[prot_pos-1]], aa_single_to_three[prot_var_seq[prot_pos-1]]
        # print(prot_r, prot_v, prot_ref, prot_var)
        if nuc_v != nuc_var or prot_r != prot_ref or prot_v != prot_var:
            print("Original annotations:", row["aa_change"], row["nt_change"])
            print(f"Actual Nuc Var:{nuc_var} | " + \
                f"Actual Prot Ref:{prot_ref}, Var:{prot_var}")
            print("========================================================")
    except:
        print(row["aa_change"], row["nt_change"], "| Protein seq len ONLY:", len(prot_ref_seq))
        print("========================================================")
    # break

## Create gene-level annotation file

In [5]:
# get unique info for genes
gene_cols = ["orf_id", "symbol", "ensembl_gene_id", "chr"]
gene_info = pd.read_table(f"{meta_inputs}/VarChamp_Consolidated_allele_collection.tsv")[gene_cols]
gene_info = gene_info.drop_duplicates()
duplicate_orfs = gene_info[gene_info.duplicated(subset=['symbol'], keep=False)]

In [6]:
# Need to figure out why there are duplicated ORF IDs
duplicate_orfs = gene_info[gene_info.duplicated(subset=['symbol'], keep=False)]

In [7]:
# write out file
gene_info.to_csv(f"{meta_outputs}/orf_info.csv", index=False)

In [9]:
gene_info

,orf_id,symbol,ensembl_gene_id,chr
0,2,GBA,ENSG00000177628,1
4,25,SLC9A3R1,ENSG00000109062,17
6,138,MYD88,ENSG00000172936,3
8,167,ACTA2,ENSG00000107796,10
12,187,FOXA3,ENSG00000170608,19
...,...,...,...,...
7991,54055,ALDH18A1,ENSG00000059573,10
8050,54326,EDA,ENSG00000158813,X
8111,56245,MAP2K1,ENSG00000169032,15
8304,100016069,KCNJ2,ENSG00000123700,17


## Pivot metadata with respect to mut_ids

In [10]:
# process metadata
wt_meta = pl.read_csv(f"{meta_outputs}/orf_info.csv")
metadata = pl.DataFrame(pd.read_csv(f"{meta_outputs}/varchamp_consolidated_alleles_clinvar_gnomad_sequence_confirmed.tsv", sep="\t"))
# mut_cols = ["mutation_id_rc4", "mutation_id_cegs2", "mutation_id_edgotyping3"]
# id_vars = [col for col in metadata.columns if col not in mut_cols]
# metadata = metadata.unpivot(index=id_vars, 
#                             on=mut_cols,
#                             variable_name="mutation_collection",
#                             value_name="mut_id")

priority_col = ["orf_id", "mut_id", "symbol", "aa_change", "nt_change", "ensembl_gene_id", "collection", "clinvar_clnsig_clean", "gnomad_af", "StarStatus"]
metadata = metadata.select(priority_col).filter(~pl.col("mut_id").is_null()).with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id"),
    pl.col("orf_id").cast(pl.Int64).alias("orf_id"),
    pl.concat_str([pl.col("symbol"), pl.col("aa_change")], separator="_").alias("allele_0")
).unique()

In [11]:
wt_meta = wt_meta.with_columns(
    pl.lit(0).cast(pl.Int64).alias("mut_id"),
    pl.col("symbol").alias("allele_0"),
    pl.lit(None).alias("aa_change"),
    pl.lit(None).alias("nt_change"),
    pl.lit(None).alias("collection"),
    pl.lit(None).alias("clinvar_clnsig_clean"),
    pl.lit(None).alias("gnomad_af"),
    pl.lit(None).alias("StarStatus")
).select(metadata.columns)

metadata = pl.concat([metadata, wt_meta])
metadata.write_csv(f"{meta_outputs}/slim_metadata.csv")